# 🧪 W2-D5 概念实验：推理优化 — KV Cache、Flash Attention、GQA

> 配套阅读：`第2周-Day5-推理优化-KV-Cache与Flash-Attention.md`（完整原理、内存分析、业务关联在那边）
> 这个 notebook 用可执行实验回答三个问题：
> 1. **KV Cache 到底省了多少计算？** 无 cache vs 有 cache 的 FLOPs 对比
> 2. **KV Cache 显存有多大？** 主流模型在不同上下文长度的内存占用
> 3. **GQA 相比 MHA 省多少？** MHA / MQA / GQA 的 KV Cache 大小对比

## 实验 1：KV Cache 的计算量节省 — 无 cache 重复计算了多少？

自回归生成第 T 个 token 时：
- **无 cache**：重新计算所有 T 个 token 的 Q, K, V → O(T·d²)
- **有 cache**：只算新 token 的 Q, K, V → O(d²)，K,V 从 cache 读取

In [ ]:
import numpy as np

n_layers = 32
n_heads = 32
d_model = 4096
d_head = d_model // n_heads

seq_lens = np.arange(1, 2049)
flops_no_cache = []
flops_with_cache = []

for T in seq_lens:
    # 无 cache：每步处理整个序列
    f_nc = n_layers * (3 * T * d_model * d_model + 2 * T * T * d_head * n_heads + T * d_model * d_model)
    flops_no_cache.append(f_nc)
    # 有 cache：QKV 投影只需处理 1 个 token
    f_wc = n_layers * (3 * 1 * d_model * d_model + 2 * T * d_head * n_heads + 1 * d_model * d_model)
    flops_with_cache.append(f_wc)

flops_no_cache = np.array(flops_no_cache)
flops_with_cache = np.array(flops_with_cache)

# 累积总 FLOPs：prefill(T=1,两者一样) + decode steps
cum_no_cache = np.cumsum(flops_no_cache)
# 有 cache: 第1步 = flops_no_cache[0], 后续第 i 步用 flops_with_cache[i]
cum_with_cache = np.zeros_like(cum_no_cache)
cum_with_cache[0] = flops_no_cache[0]
cum_with_cache[1:] = flops_no_cache[0] + np.cumsum(flops_with_cache[1:])

print(f"{'生成到长度':>10} {'无Cache(TFLOPs)':>15} {'有Cache(TFLOPs)':>15} {'节省比例':>8}")
print("-" * 55)
for T in [1, 32, 128, 512, 1024, 2048]:
    idx = T - 1
    ratio = cum_with_cache[idx] / cum_no_cache[idx]
    print(f"{T:>10} {cum_no_cache[idx]/1e12:>14.2f} {cum_with_cache[idx]/1e12:>14.2f} {ratio*100:>7.1f}%")

print(f"\n结论：生成 2048 tokens 时，KV Cache 将总计算量降到无 Cache 的不到 1/4！")

## 实验 2：KV Cache 显存占用 — 多大？能装下吗？

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

models = {
    'LLaMA-7B':   {'layers':32, 'kv_heads':32, 'd_head':128, 'dtype':2},
    'LLaMA-2-70B': {'layers':80, 'kv_heads':8,  'd_head':128, 'dtype':2},
    'Qwen2-7B':   {'layers':28, 'kv_heads':4,  'd_head':128, 'dtype':2},
    'GPT-2 XL':   {'layers':48, 'kv_heads':25, 'd_head':64,  'dtype':2},
}

seq_lens = np.array([512, 1024, 2048, 4096, 8192, 16384, 32768])

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
for idx, (name, cfg) in enumerate(models.items()):
    mems = 2 * cfg['layers'] * cfg['kv_heads'] * seq_lens * cfg['d_head'] * cfg['dtype'] / (1024**3)
    ax.plot(seq_lens, mems, 'o-', label=name, color=colors[idx], linewidth=1.5)

ax.axhline(80, color='gray', ls='--', alpha=0.5, label='A100 80GB')
ax.axhline(24, color='gray', ls=':', alpha=0.5, label='RTX 4090 24GB')
ax.set_xlabel('序列长度')
ax.set_ylabel('KV Cache 显存 (GB)')
ax.set_title('KV Cache 显存占用 vs 序列长度')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("关键数据点：")
for T in [2048, 32768]:
    mem = 2 * 32 * 32 * T * 128 * 2 / (1024**3)
    print(f"  LLaMA-7B (MHA, {T} tokens): {mem:.1f} GB")

## 实验 3：MHA vs MQA vs GQA — KV Cache 大小对比

In [ ]:
import numpy as np

n_q_heads = 32
d_head = 128
n_layers = 32
seq_len = 2048
dtype_bytes = 2

configs = {
    'MHA':  {'kv_heads': 32, 'desc': '每个 Q 头配独立 KV'},
    'GQA-8':{'kv_heads': 8,  'desc': '每 4 个 Q 头共享 1 组 KV'},
    'GQA-4':{'kv_heads': 4,  'desc': '每 8 个 Q 头共享 1 组 KV'},
    'MQA':  {'kv_heads': 1,  'desc': '所有 Q 头共享 1 组 KV'},
}

print(f"配置：Q 头数={n_q_heads}, d_head={d_head}, layers={n_layers}, seq_len={seq_len}, fp16")
print(f"\n{'方案':<8} {'KV 头数':>6} {'KV Cache (GB)':>14} {'相对MHA':>10} {'KV投影参数(M)':>14}")
print("-" * 60)

baseline_cache = None
for name, cfg in configs.items():
    n_kv = cfg['kv_heads']
    cache_gb = 2 * n_layers * n_kv * seq_len * d_head * dtype_bytes / (1024**3)
    # KV 投影参数: K_proj = d_model × (n_kv * d_head), V_proj 同理
    kv_proj_params = n_layers * 2 * (n_q_heads * d_head) * (n_kv * d_head)

    if baseline_cache is None:
        baseline_cache = cache_gb
        ratio_str = "基准"
    else:
        ratio_str = f"{cache_gb/baseline_cache*100:.0f}%"

    print(f"{name:<8} {n_kv:>6} {cache_gb:>13.2f} {ratio_str:>10} {kv_proj_params/1e6:>13.1f}")

print("\n结论：")
print("- MQA 将 KV Cache 压缩到 MHA 的 1/32，但效果下降明显")
print("- GQA-8 (LLaMA-2/3) 压缩到 1/4，效果接近 MHA — 最佳平衡点")